# MM-ViT16 — Leave-One-Tumor-Model-Out (LOTMO) Generalization

**Purpose (Reviewer #2, Comment 2):** test whether the model generalizes to a
tumor model (cell line) it never saw during training. For each cell line we
train on the **other four** lines and test on the **held-out** line. If accuracy
stays reasonable across held-out lines, the model learned generalizable
response-related features rather than cell-line-specific patterns.

### Design
- **5 folds** — hold out each of `4T1, E0771, MCA205, K7M2, B16F10` in turn.
- The held-out line is **test only** — never in train or validation. Validation
  is carved from the 4 training lines (for early stopping).
- **5 seeds per fold** → 25 trainings → mean ± 95% CI per held-out line.
- **Full multimodal MM-ViT16** (image + kPa token), identical architecture to the
  reported model and the ablation study.
- **kPa RankNormalizer fitted on the training lines only**, applied blind to the
  held-out line — a true cross-line transfer test (incl. any kPa scale shift).

### Cell-line source
Cell line is taken from sheet **`Sheet3 all day volume`** of
`clustering_all_v5.xlsx` (collaborator-confirmed as authoritative). All 1365
images have a cell line, a kPa value, and a 1/2/3 response label; every line has
all three response classes.


## 0. Environment (RunPod)

In [ ]:
# Install deps a fresh RunPod TF image is missing. Safe to re-run.
!pip install -q openpyxl scipy scikit-learn comet_ml pillow

## 1. Configuration

In [ ]:
# ─── Paths (EDIT FOR RUNPOD) ────────────────────────────────────────────────
IMAGES_DIR = "./Elastography_images"
KPA_EXCEL  = "./clustering_all_v5.xlsx"
KPA_SHEET  = "Sheet3 all day volume"     # authoritative cell-line + kPa source

# ─── Output (mirrors project layout) ────────────────────────────────────────
from pathlib import Path
OUTPUT_DIR  = Path("./lotmo_results")
DIR_TABLES  = OUTPUT_DIR / "tables"
DIR_PREDS   = OUTPUT_DIR / "predictions"
DIR_HIST    = OUTPUT_DIR / "histories"
DIR_FIG_PDF = OUTPUT_DIR / "figures" / "pdf"
DIR_FIG_PNG = OUTPUT_DIR / "figures" / "png"
DIR_CONFIG  = OUTPUT_DIR / "config"
DIR_SPLITS  = DIR_CONFIG / "splits"
DIR_MODELS  = OUTPUT_DIR / "models"
for d in (DIR_TABLES, DIR_PREDS, DIR_HIST, DIR_FIG_PDF, DIR_FIG_PNG,
          DIR_CONFIG, DIR_SPLITS, DIR_MODELS):
    d.mkdir(parents=True, exist_ok=True)

# ─── Folds = cell lines to hold out, and seeds per fold ─────────────────────
CELL_LINES = ["4T1", "E0771", "MCA205", "K7M2", "B16F10"]
SEEDS      = [42, 1, 2, 3, 4]

# ─── Validation fraction carved from the TRAINING cell lines ────────────────
VAL_FRAC = 0.15

# ─── Image / model dims (identical to the reported model) ───────────────────
IMG_HEIGHT, IMG_WIDTH = 304, 400
PATCH_SIZE            = 16
CLASS_NAMES           = ["response", "stable", "non-response"]   # maps labels 1,2,3
NUM_CLASSES           = len(CLASS_NAMES)

# ─── Transformer hyperparameters (identical to ablation/full model) ─────────
TRANSFORMER_HEADS=4; TRANSFORMER_HEAD_SIZE=32; TRANSFORMER_FF_DIM=128
TRANSFORMER_LAYERS=2; WEIGHT_DECAY=1e-4; DROPOUT_RATE=0.1
PROJECTION_DIM = TRANSFORMER_HEAD_SIZE * TRANSFORMER_HEADS

BATCH_SIZE=32; EPOCHS=100; LR_INIT=1e-3
COMET_PROJECT = "multi_modal_development"

print("Held-out folds:", CELL_LINES)
print("Seeds         :", SEEDS)
print(f"Total trainings: {len(CELL_LINES) * len(SEEDS)}")

## 2. Imports & GPU check

In [ ]:
import os, json, joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, optimizers, callbacks, Input, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, log_loss,
                             confusion_matrix)
from sklearn.preprocessing import label_binarize

gpus = tf.config.list_physical_devices("GPU")
DEVICE = "/GPU:0" if gpus else "/CPU:0"
print("TensorFlow:", tf.__version__, "| device:", DEVICE)

## 3. Load images (COMET) + kPa + cell line + labels

Images come from the Comet artifact (same as the ViT16 notebooks). kPa, cell
line, and label come from the `Sheet3 all day volume` sheet, joined on filename.

In [ ]:
# ─── Comet experiment + image artifact ──────────────────────────────────────
from comet_ml import Experiment
experiment = Experiment(api_key=os.getenv("COMET_API_KEY"),
                        project_name=COMET_PROJECT)
logged_artifact = experiment.get_artifact("elastography_images_merged")
if not os.path.exists("./Elastography_images"):
    logged_artifact.download("./")

In [ ]:
# ─── Load all images ────────────────────────────────────────────────────────
class_names = logged_artifact.metadata["classes"]
assert class_names == CLASS_NAMES, f"artifact classes {class_names} != {CLASS_NAMES}"
dataset = ImageDataGenerator().flow_from_directory(
    "./Elastography_images", batch_size=1579, class_mode="sparse",
    target_size=(IMG_HEIGHT, IMG_WIDTH), shuffle=False, classes=class_names)
with tf.device(DEVICE):
    x_all, y_all = dataset.next()
basenames = [os.path.basename(p) for p in dataset.filenames]
print("Classes:", class_names, "| images:", x_all.shape[0])

In [ ]:
# ─── Join kPa + cell line + label from the authoritative sheet ──────────────
df = pd.read_excel(KPA_EXCEL, sheet_name=KPA_SHEET)
lab_col = "Respone/stable/non-Response"
kpa_col = "Elastic Modulus SWE (kPa)"
df = df[["name", lab_col, kpa_col, "Cell lines"]].copy()
df["name"] = df["name"].astype(str).str.replace("\u03A4", "T", regex=False)

kpa_map  = dict(zip(df["name"], df[kpa_col]))
line_map = dict(zip(df["name"], df["Cell lines"]))
# Sheet labels are 1/2/3; image folders give 0/1/2 in CLASS_NAMES order
# (response/stable/non-response). Sheet 1->response(0), 2->stable(1), 3->non(2).
lab_map  = dict(zip(df["name"], df[lab_col]))

# Keep only images present in the sheet with a cell line + kPa + label
keep, X_img, X_num, Y, CL, FN = [], [], [], [], [], []
missing = 0
for i, b in enumerate(basenames):
    if b in line_map and pd.notna(line_map[b]) and b in kpa_map and pd.notna(kpa_map[b]) and b in lab_map and pd.notna(lab_map[b]):
        X_img.append(x_all[i]); X_num.append([float(kpa_map[b])])
        Y.append(int(lab_map[b]) - 1)          # 1/2/3 -> 0/1/2
        CL.append(str(line_map[b])); FN.append(b)
    else:
        missing += 1

x_all   = np.asarray(X_img, dtype=np.float32)
X_num_all = np.asarray(X_num, dtype=np.float32)
y_all   = np.asarray(Y, dtype=np.int64)
cellline_all = np.asarray(CL)
basenames_arr = np.asarray(FN)

print(f"Aligned {len(y_all)} images ({missing} dropped: no cell line/kPa/label in sheet).")
print("Per cell line:", {c: int((cellline_all==c).sum()) for c in CELL_LINES})
print("Label check (0/1/2):", {CLASS_NAMES[k]: int((y_all==k).sum()) for k in range(NUM_CLASSES)})
assert set(np.unique(y_all)) <= {0,1,2}, "labels must be 0/1/2 after the -1 shift"

## 4. Helpers: LOTMO split, kPa normalizer, data pipeline

In [ ]:
# ─── kPa rank-normalizer (fit on TRAIN lines only) ──────────────────────────
class RankNormalizer:
    def __init__(self): self.train_sorted = None
    def fit(self, X):
        self.train_sorted = np.sort(np.asarray(X).flatten()); return self
    def transform(self, X):
        arr = np.asarray(X).flatten()
        ranks = np.searchsorted(self.train_sorted, arr, side="right").astype(np.float64)
        return ((ranks - 0.5) / len(self.train_sorted)).reshape(-1,1).astype(np.float32)
    def fit_transform(self, X): return self.fit(X).transform(X)

In [ ]:
# ─── Leave-one-tumor-model-out split ────────────────────────────────────────
# TRAIN+VAL = all images NOT from `held_out`; TEST = all images from `held_out`.
# Validation is a stratified slice of the training lines (held_out never leaks in).
def make_lotmo_split(held_out, seed):
    is_test = (cellline_all == held_out)
    tr_idx_all = np.where(~is_test)[0]
    te_idx     = np.where(is_test)[0]

    # carve a stratified validation set out of the training-line indices
    tr_idx, va_idx = train_test_split(
        tr_idx_all, test_size=VAL_FRAC, random_state=seed,
        shuffle=True, stratify=y_all[tr_idx_all])

    def pick(idx):
        return (x_all[idx], X_num_all[idx], y_all[idx], basenames_arr[idx])
    Xi_tr,Xn_tr,Y_tr,fn_tr = pick(tr_idx)
    Xi_va,Xn_va,Y_va,fn_va = pick(va_idx)
    Xi_te,Xn_te,Y_te,fn_te = pick(te_idx)
    return dict(Xi_tr=Xi_tr,Xn_tr=Xn_tr,Y_tr=Y_tr,fn_tr=fn_tr,
                Xi_va=Xi_va,Xn_va=Xn_va,Y_va=Y_va,fn_va=fn_va,
                Xi_te=Xi_te,Xn_te=Xn_te,Y_te=Y_te,fn_te=fn_te,
                held_out=held_out)

In [ ]:
# ─── tf.data pipeline (identical to ablation) ───────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE
def _preprocess(image, numeric, label):
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = tf.clip_by_value(tf.cast(image, tf.float32)/255.0, 0.0, 1.0)
    return (image, tf.cast(numeric, tf.float32)), tf.cast(label, tf.int64)
_aug = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.05),
                         layers.RandomTranslation(0.05,0.05), layers.RandomContrast(0.05)],
                        name="data_augmentation")
def _augment(inputs, label):
    image, numeric = inputs
    return (tf.clip_by_value(_aug(image),0.0,1.0), numeric), label
def make_dataset(images, numerics, labels, seed, shuffle=False, augment=False, cache=False):
    with tf.device("/CPU:0"):
        ds = tf.data.Dataset.from_tensor_slices((images, numerics, labels))
    if shuffle: ds = ds.shuffle(len(images), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(_preprocess, num_parallel_calls=AUTOTUNE)
    if augment: ds = ds.map(_augment, num_parallel_calls=AUTOTUNE)
    if cache: ds = ds.cache()
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

## 5. Model — full multimodal MM-ViT16 (identical to the reported model)

In [ ]:
class TokenSlice(layers.Layer):
    def __init__(self, index=0, **kw): super().__init__(**kw); self.index=index
    def call(self, x): return x[:, self.index, :]
    def get_config(self): return {**super().get_config(), "index": self.index}

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout_rate=0.1):
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout_rate)(x, x)
    x = layers.Dropout(dropout_rate)(x); res = layers.Add()([inputs, x])
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.Dense(ff_dim, activation="relu")(x); x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(inputs.shape[-1])(x)
    return layers.Add()([res, x])

def build_full_model():
    image_input = Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="image_input")
    num_input   = Input(shape=(1,), name="num_input")
    x = layers.Conv2D(PROJECTION_DIM, (PATCH_SIZE,PATCH_SIZE), strides=(PATCH_SIZE,PATCH_SIZE),
                      padding="valid", kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                      name="patch_proj")(image_input)
    num_patches = x.shape[1]*x.shape[2]
    patch_tokens = layers.Reshape((num_patches, PROJECTION_DIM), name="patch_tokens")(x)
    num_proj  = layers.Dense(PROJECTION_DIM, activation="relu",
                             kernel_regularizer=regularizers.l2(WEIGHT_DECAY), name="num_proj")(num_input)
    num_token = layers.Reshape((1, PROJECTION_DIM), name="num_token")(num_proj)
    seq = layers.Concatenate(axis=1, name="token_concat")([num_token, patch_tokens])
    L = 1 + num_patches
    pos = layers.Embedding(L, PROJECTION_DIM, name="pos_embedding")
    seq = layers.Add(name="add_pos")([seq, pos(tf.range(L)[tf.newaxis,:])])
    for _ in range(TRANSFORMER_LAYERS):
        seq = transformer_encoder(seq, TRANSFORMER_HEAD_SIZE, TRANSFORMER_HEADS, TRANSFORMER_FF_DIM, DROPOUT_RATE)
    cls = TokenSlice(0, name="cls_pick")(seq)
    cls = layers.LayerNormalization(epsilon=1e-6, name="cls_norm")(cls)
    h = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(WEIGHT_DECAY), name="cls_embedding")(cls)
    h = layers.Dropout(DROPOUT_RATE, name="cls_dropout")(h)
    out = layers.Dense(NUM_CLASSES, activation="softmax", kernel_regularizer=regularizers.l2(WEIGHT_DECAY), name="predictions")(h)
    return Model([image_input, num_input], out, name="mmvit16_full")

## 6. Train + evaluate one (held-out line, seed)

In [ ]:
def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"]=str(seed)
    import random as _r; _r.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    try: tf.config.experimental.enable_op_determinism()
    except Exception: pass

def compute_metrics(y_true, y_prob):
    y_pred = y_prob.argmax(axis=1)
    yb = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    out = {}
    for k, fn in [("accuracy", lambda: accuracy_score(y_true, y_pred)),
                  ("macro_f1", lambda: f1_score(y_true, y_pred, average="macro")),
                  ("roc_auc_ovr", lambda: roc_auc_score(yb, y_prob, average="macro", multi_class="ovr")),
                  ("log_loss", lambda: log_loss(y_true, y_prob, labels=list(range(NUM_CLASSES))))]:
        try: out[k] = fn()
        except Exception: out[k] = float("nan")
    return out

def run_fold_seed(held_out, seed):
    set_all_seeds(seed)
    sp = make_lotmo_split(held_out, seed)
    norm = RankNormalizer()
    Xn_tr = norm.fit_transform(sp["Xn_tr"])        # fit on TRAIN lines only
    Xn_va = norm.transform(sp["Xn_va"])
    Xn_te = norm.transform(sp["Xn_te"])            # held-out line, blind

    train_set = make_dataset(sp["Xi_tr"], Xn_tr, sp["Y_tr"], seed, shuffle=True, augment=True)
    val_set   = make_dataset(sp["Xi_va"], Xn_va, sp["Y_va"], seed, cache=True)
    test_set  = make_dataset(sp["Xi_te"], Xn_te, sp["Y_te"], seed, cache=True)

    keras.backend.clear_session(); set_all_seeds(seed)
    model = build_full_model()
    cosine = optimizers.schedules.CosineDecay(initial_learning_rate=LR_INIT, decay_steps=EPOCHS)
    model.compile(optimizer=optimizers.SGD(learning_rate=LR_INIT, momentum=0.9, nesterov=True),
                  loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
    cbs = [callbacks.LearningRateScheduler(lambda e, lr: float(cosine(e))),
           callbacks.EarlyStopping(monitor="val_loss", patience=20,
                                   restore_best_weights=True, start_from_epoch=20)]
    with tf.device(DEVICE):
        history = model.fit(train_set, validation_data=val_set, epochs=EPOCHS, callbacks=cbs, verbose=2)

    y_prob = model.predict(test_set, verbose=0); y_true = sp["Y_te"].astype(int)
    y_pred = y_prob.argmax(axis=1)
    m = compute_metrics(y_true, y_prob)
    m.update({"held_out": held_out, "seed": seed,
              "n_test": int(len(y_true)), "n_train": int(len(sp["Y_tr"]))})
    print(f"  [held_out={held_out} | seed {seed}] acc={m['accuracy']:.3f} "
          f"macroF1={m['macro_f1']:.3f} auc={m['roc_auc_ovr']:.3f} (n_test={m['n_test']})")

    tag = f"{held_out}_seed{seed}"
    pd.DataFrame({"filename": sp["fn_te"], "y_true": y_true, "y_pred": y_pred,
                  **{f"p_{CLASS_NAMES[c]}": y_prob[:,c] for c in range(NUM_CLASSES)}}
                 ).to_csv(DIR_PREDS / f"preds_{tag}.csv", index=False)
    with open(DIR_HIST / f"history_{tag}.json", "w") as f:
        json.dump({k:[float(v) for v in vv] for k,vv in history.history.items()}, f)
    model.save(DIR_MODELS / f"model_{tag}.keras")
    with open(DIR_SPLITS / f"split_{tag}.json", "w") as f:
        json.dump({"held_out":held_out,"seed":seed,
                   "train":list(map(str,sp["fn_tr"])),"val":list(map(str,sp["fn_va"])),
                   "test":list(map(str,sp["fn_te"]))}, f)
    return m

## 7. Run the full grid (held-out line × seed)

Checkpointed to CSV after each run; resumes if interrupted.

In [ ]:
RESULTS_CSV = DIR_TABLES / "lotmo_runs.csv"
done = set(); records = []
if RESULTS_CSV.exists():
    prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(prev["held_out"], prev["seed"])); records = prev.to_dict("records")
    print(f"Resuming — {len(done)} runs already complete.")

for held_out in CELL_LINES:
    for seed in SEEDS:
        if (held_out, seed) in done:
            print(f"skip {held_out} seed {seed} (done)"); continue
        print(f"=== TRAIN  held_out={held_out}  seed={seed} ===")
        records.append(run_fold_seed(held_out, seed))
        pd.DataFrame(records).to_csv(RESULTS_CSV, index=False)

results_df = pd.DataFrame(records)
print("\nAll runs complete.")
results_df

## 8. Per-fold summary: mean ± 95% CI

This is the table for the rebuttal — accuracy on each unseen cell line.

In [ ]:
from scipy import stats as sp_stats
METRICS = ["accuracy", "macro_f1", "roc_auc_ovr", "log_loss"]
def mean_ci(vals, conf=0.95):
    vals = np.asarray(vals, dtype=float); n = len(vals); mean = vals.mean()
    if n < 2: return mean, np.nan, np.nan
    half = sp_stats.sem(vals) * sp_stats.t.ppf((1+conf)/2.0, n-1)
    return mean, mean-half, mean+half

rows = []
for line in CELL_LINES:
    sub = results_df[results_df["held_out"]==line]
    r = {"held_out": line, "n_seeds": len(sub),
         "n_test": int(sub["n_test"].iloc[0]) if len(sub) else 0}
    for met in METRICS:
        mean, lo, hi = mean_ci(sub[met].values)
        r[met] = f"{mean:.3f} ± {(hi-lo)/2:.3f}" if np.isfinite(hi) else f"{mean:.3f}"
    rows.append(r)
# overall row (mean across folds of per-fold means)
overall = {"held_out": "MEAN (all folds)", "n_seeds": "", "n_test": int(results_df["n_test"].sum()/len(SEEDS))}
for met in METRICS:
    per_fold = [results_df[results_df["held_out"]==l][met].mean() for l in CELL_LINES]
    overall[met] = f"{np.mean(per_fold):.3f} ± {np.std(per_fold):.3f}"
rows.append(overall)
summary_df = pd.DataFrame(rows)
summary_df.to_csv(DIR_TABLES / "lotmo_summary.csv", index=False)
summary_df

## 9. Save run configuration snapshot

In [ ]:
run_config = {"cell_lines": CELL_LINES, "seeds": SEEDS, "val_frac": VAL_FRAC,
    "img_height": IMG_HEIGHT, "img_width": IMG_WIDTH, "patch_size": PATCH_SIZE,
    "class_names": CLASS_NAMES, "kpa_sheet": KPA_SHEET,
    "transformer": {"heads": TRANSFORMER_HEADS, "head_size": TRANSFORMER_HEAD_SIZE,
                    "ff_dim": TRANSFORMER_FF_DIM, "layers": TRANSFORMER_LAYERS,
                    "weight_decay": WEIGHT_DECAY, "dropout": DROPOUT_RATE},
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr_init": LR_INIT, "tf_version": tf.__version__}
with open(DIR_CONFIG / "run_config.json", "w") as f: json.dump(run_config, f, indent=2)
print("Saved run_config.json"); run_config

## 10. Visualizations

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc
PALETTE = {"4T1":"#1b7837","E0771":"#762a83","MCA205":"#d6604d","K7M2":"#4393c3","B16F10":"#e08214"}
def savefig(fig, name):
    fig.savefig(DIR_FIG_PDF / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(DIR_FIG_PNG / f"{name}.png", dpi=200, bbox_inches="tight"); plt.close(fig)
PROB_COLS = [f"p_{c}" for c in CLASS_NAMES]

### 10.1 Per-fold accuracy & macro-F1 (held-out cell line, 95% CI)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, met, title in zip(axes, ["accuracy","macro_f1"], ["Accuracy","Macro-F1"]):
    means, errs, cols = [], [], []
    for line in CELL_LINES:
        vals = results_df[results_df["held_out"]==line][met].dropna().values
        mean, lo, hi = mean_ci(vals); means.append(mean)
        errs.append((hi-lo)/2 if np.isfinite(hi) else 0); cols.append(PALETTE[line])
    xp = np.arange(len(CELL_LINES))
    ax.bar(xp, means, yerr=errs, capsize=5, color=cols, edgecolor="black", linewidth=0.6, alpha=0.9)
    ax.axhline(np.mean(means), ls="--", color="gray", label=f"mean={np.mean(means):.3f}")
    ax.set_xticks(xp); ax.set_xticklabels(CELL_LINES, rotation=15)
    ax.set_ylim(0,1.0); ax.set_ylabel(title); ax.set_title(f"{title} on held-out cell line")
    ax.grid(axis="y", alpha=0.3); ax.legend()
    for x,m_ in zip(xp,means): ax.text(x, m_+0.02, f"{m_:.3f}", ha="center", fontsize=9)
fig.suptitle("LOTMO generalization — each bar = a tumor model the model never trained on", y=1.02)
fig.tight_layout(); savefig(fig, "fig1_lotmo_per_fold"); fig

### 10.2 Per-seed spread per held-out line

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
data = [results_df[results_df["held_out"]==l]["accuracy"].dropna().values for l in CELL_LINES]
bp = ax.boxplot(data, widths=0.5, patch_artist=True, showmeans=True)
for patch, line in zip(bp["boxes"], CELL_LINES): patch.set_facecolor(PALETTE[line]); patch.set_alpha(0.6)
for i, line in enumerate(CELL_LINES):
    ys = results_df[results_df["held_out"]==line]["accuracy"].dropna().values
    ax.scatter(np.full(len(ys), i+1), ys, color="black", s=18, zorder=3, alpha=0.7)
ax.set_xticks(np.arange(1,len(CELL_LINES)+1)); ax.set_xticklabels(CELL_LINES, rotation=15)
ax.set_ylabel("Accuracy"); ax.set_title("Per-seed accuracy on each held-out cell line")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); savefig(fig, "fig2_lotmo_boxplots"); fig

### 10.3 Confusion matrices per held-out line (pooled over seeds)

In [ ]:
fig, axes = plt.subplots(1, len(CELL_LINES), figsize=(4.0*len(CELL_LINES), 4))
for ax, line in zip(axes, CELL_LINES):
    yt, yp = [], []
    for seed in SEEDS:
        fp = DIR_PREDS / f"preds_{line}_seed{seed}.csv"
        if fp.exists(): d=pd.read_csv(fp); yt+=list(d["y_true"]); yp+=list(d["y_pred"])
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES))).astype(float)
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    ax.imshow(cmn, cmap="Greens", vmin=0, vmax=1); ax.set_title(line, fontsize=11)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=8)
    ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j,i,f"{cmn[i,j]:.2f}",ha="center",va="center",
                    color="white" if cmn[i,j]>0.5 else "black", fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
fig.suptitle("Confusion on held-out cell line (row-normalized, pooled over seeds)", y=1.04)
fig.tight_layout(); savefig(fig, "fig3_lotmo_confusion"); fig

## 11. Interpreting the result (for the rebuttal)

- **Each bar/fold is a tumor model the network never saw during training** — so the
  accuracy there is pure cross-line generalization.
- **Consistent accuracy across folds** → the model learned generalizable
  response-related features, not cell-line-specific shortcuts (directly answers
  Reviewer #2, Comment 2).
- A weak fold (one cell line much lower) flags a tumor type whose mechanics differ;
  worth discussing rather than hiding.
- **Caveat:** this is leave-one-CELL-LINE-out. It does not by itself control for
  animal-level grouping *within* the training lines; combine with the animal-grouped
  split (Comment 1) for the strongest claim. The held-out line is fully unseen, so
  the test fold itself has no animal leakage with training by construction.

Numbers are mean ± 95% CI over 5 seeds per held-out line.


In [ ]:
# Output manifest
import os
print("MANIFEST —", OUTPUT_DIR.resolve())
tot=0
for root,_,files in os.walk(OUTPUT_DIR):
    for fn in sorted(files):
        p=Path(root)/fn; tot+=p.stat().st_size
        print(f"  {p.stat().st_size/1024:8.1f} KB  {p.relative_to(OUTPUT_DIR)}")
print(f"Total: {tot/1024/1024:.1f} MB")